# 23. Mask R-CNN 핵심 아이디어

이 노트북은 `22_Instance_Segmentation_개념.ipynb` 다음 단계로, instance segmentation의 대표 모델인 Mask R-CNN의 핵심 구조를 이해하는 것이 목표입니다.

Mask R-CNN은 Faster R-CNN의 객체 탐지 흐름에 **mask branch**를 추가한 모델입니다. 먼저 객체 후보 영역을 찾고, 각 후보 영역마다 class, box 보정값, mask를 함께 예측합니다.

이번 노트북의 목표는 다음과 같습니다.

- Mask R-CNN이 detection과 segmentation을 어떻게 결합하는지 이해합니다.
- Faster R-CNN 구조에서 mask branch가 추가되는 위치를 파악합니다.
- RoI Pooling 대신 RoI Align이 필요한 이유를 직관적으로 확인합니다.
- instance별 binary mask가 최종 결과로 어떻게 정리되는지 이해합니다.

## 23-1. 준비

외부 데이터나 사전학습 가중치 없이 실행되도록 작은 feature map과 예제 RoI를 직접 만듭니다. 실제 Mask R-CNN 구현은 복잡하지만, 여기서는 구조를 이해하는 데 필요한 핵심 흐름만 단순화해서 다룹니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

## 23-2. Mask R-CNN의 큰 흐름

Mask R-CNN은 한 번에 픽셀 전체 class map을 예측하는 semantic segmentation 모델과 다르게, 먼저 객체 후보를 찾은 뒤 후보마다 mask를 만듭니다.

```text
image
  -> backbone CNN
  -> feature map
  -> RPN으로 RoI 후보 생성
  -> RoI Align으로 후보 영역 feature 추출
  -> class head / box head / mask head
  -> instance별 class, bbox, mask
```

핵심은 **mask를 이미지 전체에 바로 그리는 것이 아니라, 각 RoI 안에서 작은 binary mask를 예측한 뒤 원본 위치에 다시 붙인다**는 점입니다.

In [ ]:
steps = [
    'image',
    'backbone\nfeature map',
    'RPN\nRoI 후보',
    'RoI Align\n고정 크기 feature',
    'class / box\nmask heads',
    'instance\n결과',
]

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.set_xlim(0, len(steps))
ax.set_ylim(0, 1)
ax.axis('off')

for i, step in enumerate(steps):
    ax.add_patch(Rectangle((i + 0.08, 0.25), 0.82, 0.5, facecolor='#f2f2f2', edgecolor='#333333'))
    ax.text(i + 0.49, 0.5, step, ha='center', va='center', fontsize=11)
    if i < len(steps) - 1:
        ax.annotate('', xy=(i + 1.03, 0.5), xytext=(i + 0.9, 0.5), arrowprops=dict(arrowstyle='->'))

plt.title('Mask R-CNN 전체 흐름')
plt.show()

## 23-3. Detection 결과에 mask branch를 추가한다

Faster R-CNN은 각 RoI에 대해 class와 bounding box를 예측합니다. Mask R-CNN은 여기에 mask branch를 하나 더 붙입니다.

- class head: 이 RoI가 어떤 클래스인지 예측합니다.
- box head: RoI 위치를 더 정확하게 보정합니다.
- mask head: RoI 내부에서 객체가 차지하는 픽셀을 예측합니다.

따라서 Mask R-CNN의 출력은 보통 `class`, `score`, `box`, `mask`를 함께 가집니다.

In [ ]:
detections = [
    {'class': 'person', 'score': 0.94, 'box': (22, 15, 48, 72), 'color': '#54a24b'},
    {'class': 'car', 'score': 0.89, 'box': (58, 36, 108, 70), 'color': '#f58518'},
]

canvas = np.ones((90, 130, 3), dtype=float)
canvas[62:, :, :] = np.array([0.72, 0.72, 0.72])
canvas[:62, :, :] = np.array([0.82, 0.90, 1.00])

fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(canvas)
for det in detections:
    x1, y1, x2, y2 = det['box']
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=det['color'], linewidth=2))
    ax.text(x1, y1 - 3, f"{det['class']} {det['score']:.2f}", color=det['color'], fontsize=11, weight='bold')
ax.set_title('Detection head가 만드는 class와 box')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

## 23-4. RoI Align이 필요한 이유

RoI 후보는 원본 이미지 좌표에서는 소수점 좌표를 가질 수 있습니다. 하지만 feature map은 이미지보다 작고 격자 형태입니다. RoI Pooling처럼 좌표를 거칠게 반올림하면 작은 위치 차이가 mask 경계에 크게 영향을 줄 수 있습니다.

RoI Align은 좌표를 강제로 반올림하지 않고, 주변 feature 값을 보간해서 더 정밀한 RoI feature를 만듭니다. 아래 예제는 같은 RoI를 거친 방식과 보간 방식으로 읽을 때 값이 달라질 수 있음을 보여줍니다.

In [ ]:
feature_map = np.arange(1, 65, dtype=float).reshape(8, 8)
roi = (1.4, 1.2, 6.6, 5.7)  # x1, y1, x2, y2 in feature-map coordinates

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(feature_map, cmap='viridis')
for y in range(feature_map.shape[0]):
    for x in range(feature_map.shape[1]):
        ax.text(x, y, int(feature_map[y, x]), ha='center', va='center', color='white', fontsize=9)

x1, y1, x2, y2 = roi
ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red', linewidth=2))
ax.set_title('feature map 위의 소수점 RoI')
ax.set_xticks(range(8))
ax.set_yticks(range(8))
plt.show()

In [ ]:
def nearest_sample(feature, xs, ys):
    xs = np.clip(np.rint(xs).astype(int), 0, feature.shape[1] - 1)
    ys = np.clip(np.rint(ys).astype(int), 0, feature.shape[0] - 1)
    return feature[ys, xs]


def bilinear_sample(feature, xs, ys):
    h, w = feature.shape
    xs = np.clip(xs, 0, w - 1)
    ys = np.clip(ys, 0, h - 1)
    x0 = np.floor(xs).astype(int)
    x1 = np.clip(x0 + 1, 0, w - 1)
    y0 = np.floor(ys).astype(int)
    y1 = np.clip(y0 + 1, 0, h - 1)

    wa = (x1 - xs) * (y1 - ys)
    wb = (xs - x0) * (y1 - ys)
    wc = (x1 - xs) * (ys - y0)
    wd = (xs - x0) * (ys - y0)

    return (
        feature[y0, x0] * wa
        + feature[y0, x1] * wb
        + feature[y1, x0] * wc
        + feature[y1, x1] * wd
    )


def roi_align_demo(feature, roi, output_size=4):
    x1, y1, x2, y2 = roi
    xs = np.linspace(x1, x2, output_size, endpoint=False) + (x2 - x1) / output_size / 2
    ys = np.linspace(y1, y2, output_size, endpoint=False) + (y2 - y1) / output_size / 2
    grid_x, grid_y = np.meshgrid(xs, ys)
    pooled_nearest = nearest_sample(feature, grid_x, grid_y)
    pooled_align = bilinear_sample(feature, grid_x, grid_y)
    return pooled_nearest, pooled_align

nearest, aligned = roi_align_demo(feature_map, roi)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(nearest, cmap='viridis')
axes[0].set_title('거친 샘플링')
axes[1].imshow(aligned, cmap='viridis')
axes[1].set_title('RoI Align 방식 보간')
for ax, arr in zip(axes, [nearest, aligned]):
    for y in range(arr.shape[0]):
        for x in range(arr.shape[1]):
            ax.text(x, y, f'{arr[y, x]:.1f}', ha='center', va='center', color='white', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## 23-5. Mask head는 RoI 안의 binary mask를 예측한다

Mask head의 출력은 RoI마다 작은 크기의 mask입니다. 실제 Mask R-CNN은 보통 class별 mask logit을 만들고, 최종 class에 해당하는 mask를 선택합니다.

아래에서는 각 box 안에서 간단한 타원 mask를 만들어, RoI 내부 mask가 원본 이미지 위치로 어떻게 배치되는지 확인합니다.

In [ ]:
def make_ellipse_mask(height, width, margin=0.12):
    yy, xx = np.mgrid[:height, :width]
    cx, cy = (width - 1) / 2, (height - 1) / 2
    rx = width * (0.5 - margin)
    ry = height * (0.5 - margin)
    return (((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1).astype(float)

mask_canvas = np.zeros((90, 130), dtype=float)
instance_masks = []
for idx, det in enumerate(detections, start=1):
    x1, y1, x2, y2 = map(int, det['box'])
    h, w = y2 - y1, x2 - x1
    mask = make_ellipse_mask(h, w)
    mask_canvas[y1:y2, x1:x2] = np.maximum(mask_canvas[y1:y2, x1:x2], mask * idx)
    instance_masks.append(mask)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(instance_masks[0], cmap='Greens')
axes[0].set_title('person RoI mask')
axes[1].imshow(instance_masks[1], cmap='Oranges')
axes[1].set_title('car RoI mask')
axes[2].imshow(mask_canvas, cmap=ListedColormap(['#ffffff', '#54a24b', '#f58518']), vmin=0, vmax=2)
axes[2].set_title('원본 위치에 배치한 instance masks')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## 23-6. 최종 결과 형식 읽기

실제 라이브러리에서 Mask R-CNN 결과를 읽을 때도 핵심은 같습니다. 각 instance가 class, score, box, mask를 하나씩 가집니다.

```python
{
    'class': 'person',
    'score': 0.94,
    'box': (x1, y1, x2, y2),
    'mask': binary_mask,
}
```

mask는 보통 확률값으로 나오기 때문에 threshold를 적용해서 binary mask로 바꿉니다. threshold가 높으면 더 확실한 영역만 남고, 낮으면 mask가 넓어질 수 있습니다.

In [ ]:
prob_mask = np.zeros((60, 60), dtype=float)
yy, xx = np.mgrid[:60, :60]
prob_mask = np.exp(-(((xx - 30) / 15) ** 2 + ((yy - 30) / 18) ** 2))

thresholds = [0.3, 0.5, 0.7]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(prob_mask, cmap='magma', vmin=0, vmax=1)
axes[0].set_title('mask probability')
for ax, threshold in zip(axes[1:], thresholds):
    ax.imshow(prob_mask >= threshold, cmap='gray')
    ax.set_title(f'threshold={threshold}')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## 23-7. Mask R-CNN의 장점과 한계

Mask R-CNN의 장점은 detection 결과와 instance mask를 함께 얻을 수 있다는 점입니다. 사람 수, 차량 수, 개별 물체 영역처럼 **객체 단위의 구분**이 중요한 문제에 잘 맞습니다.

다만 구조가 여러 단계로 나뉘어 있어 one-stage detector나 semantic segmentation 모델보다 추론이 무거울 수 있습니다. 또한 RoI 후보가 잘못 잡히면 mask 품질도 함께 나빠질 수 있습니다.

실전에서는 다음 질문을 먼저 확인합니다.

- 개별 객체를 세야 하는가?
- 같은 클래스의 객체를 서로 분리해야 하는가?
- bbox만으로 부족하고 정확한 object contour가 필요한가?

이 질문에 답이 `예`라면 Mask R-CNN 같은 instance segmentation 모델을 고려할 수 있습니다.

## 정리

- Mask R-CNN은 Faster R-CNN에 mask branch를 추가한 instance segmentation 모델입니다.
- RoI Align은 후보 영역 feature를 더 정밀하게 추출하기 위한 핵심 구성입니다.
- 최종 출력은 instance마다 `class`, `score`, `box`, `mask`를 함께 가집니다.
- semantic segmentation은 클래스별 영역을, Mask R-CNN은 개별 객체별 영역을 다루는 데 초점이 있습니다.

다음 노트북 `24_사전학습_Segmentation_모델_실습.ipynb`에서는 torchvision의 segmentation 모델을 기준으로 실제 모델 결과를 읽고 시각화하는 흐름을 다룹니다.